# Clasificación de 3 escenas con CNN

Se entrenan dos variantes de red convolucional **secuencial** sobre el dataset
local de 3 escenas (`coast`, `forest`, `highway`; 961 imágenes: 361 de costa,
329 de bosque y 271 de autopista, es decir, un reparto aproximado de 38/34/28 %)
y cada una guarda sus resultados en una carpeta independiente:

| Variante | Carpeta | Descripción |
|---|---|---|
| CNN base | `resultados/cnn/base/` | Réplica del cuaderno del profesor: `Rescaling(1/255)` y tres bloques de 8 → 16 → 32 filtros, `Flatten` + `Dense(3, softmax)`. |
| CNN mejorada | `resultados/cnn/mejorada/` | Más filtros (32-64-128), `BatchNormalization`, `GlobalAveragePooling`, `Dropout`, augmentación y `class_weight` para el desbalance. |

Las imágenes se redimensionan a 128×128 y se cargan **sin normalizar**: la capa
`Rescaling(1/255)` vive dentro de los dos modelos, igual que en el cuaderno
base del profesor. El split es el común del reto: 75 % train, 15 % validación y
25 % test, estratificado con semilla 20 (`resultados/split.json`). El test
**no** se usa para early stopping.

Los modelos ya entrenados se cargan por defecto desde `resultados/`. Para volver
a entrenarlos, cambia `FORCE_RETRAIN = True` en la celda de configuración.

## Configuración

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from keras import layers
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.layers import (
    Activation,
    BatchNormalization,
    Conv2D,
    Dense,
    Dropout,
    Flatten,
    GlobalAveragePooling2D,
    Input,
    MaxPooling2D,
    Rescaling,
)
from keras.models import Sequential
from keras.optimizers import Adam

import common_3scenes as cs

FORCE_RETRAIN = False
sns.set_theme(style="whitegrid", context="notebook")
keras.utils.set_random_seed(cs.SEED)
np.random.seed(cs.SEED)

## Carga del dataset y split común

In [ ]:
data = cs.load_split_data()
X_train = data["X_train"].astype("float32")
X_val = data["X_val"].astype("float32")
X_test = data["X_test"].astype("float32")
y_train_labels = data["y_train"]
y_val_labels = data["y_val"]
y_test_labels = data["y_test"]
test_paths = data["test_paths"]

y_train = cs.one_hot(y_train_labels)
y_val = cs.one_hot(y_val_labels)

# Class weights for the unbalanced training set (approx. 38/34/28 %).
class_counts = {name: int(np.sum(y_train_labels == name)) for name in cs.CLASSES}
total_train = sum(class_counts.values())
class_weight = {
    index: total_train / (len(cs.CLASSES) * class_counts[name])
    for index, name in enumerate(cs.CLASSES)
}

print("Train:", X_train.shape, "| Validación:", X_val.shape, "| Test:", X_test.shape)
print("Distribución de test:", {name: int(np.sum(y_test_labels == name)) for name in cs.CLASSES})
print("Distribución de train:", class_counts)
print(
    "class_weight:",
    {name: round(class_weight[index], 3) for index, name in enumerate(cs.CLASSES)},
)

## Arquitecturas

La **CNN base** replica la arquitectura del cuaderno del profesor
(`3scenesCNN.ipynb`): entrada de 128×128×3, `Rescaling(1/255)` y tres bloques
`Conv2D(3×3, padding="same", activation="relu") + MaxPooling2D` con 8, 16 y 32
filtros, seguidos de `Flatten()` y `Dense(3, activation="softmax")`. Se entrena
15 épocas con `Adam(1e-3)`, `categorical_crossentropy` y batch de 32, sin
augmentación ni callbacks.

La **CNN mejorada** mantiene la estructura "bloques convolucionales → cabeza
densa" y añade lo que a la base le falta:

* Bloques `Conv2D(f, 3×3, padding="same", use_bias=False) → BatchNormalization → ReLU → MaxPooling2D` con `f` = 32, 64 y 128.
* Augmentación **solo durante el entrenamiento**: `RandomFlip("horizontal")`, `RandomRotation(0.08)`, `RandomZoom(0.1)` y `RandomContrast(0.1)`. Las capas viven **dentro del modelo** y Keras las desactiva automáticamente en inferencia.
* Cabeza ligera: `GlobalAveragePooling2D() → Dropout(0.4) → Dense(3, softmax)`.
* `class_weight` para compensar el desbalance del train.
* `EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True)` y `ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5)` con un máximo de 60 épocas.

### Nota sobre `Adam(learning_rate=1e-3, decay=1e-3/50)` y Keras 3

El cuaderno original del profesor compila el optimizador como
`Adam(learning_rate=1e-3, decay=1e-3/50)`. El argumento `decay` **fue eliminado
en Keras 3**: los schedules de learning rate ahora se pasan como objetos (por
ejemplo `keras.optimizers.schedules.ExponentialDecay`) o se gestionan con
callbacks como `ReduceLROnPlateau`.

Decisión: usar `Adam(learning_rate=1e-3)` sin decay en la CNN base, para
replicar la arquitectura y el flujo del cuaderno original con la API actual de
Keras (es una decisión defendible en la defensa oral). La CNN mejorada controla
el learning rate con `ReduceLROnPlateau`, que cumple el mismo papel de reducir
el paso de aprendizaje a medida que avanza el entrenamiento.

In [ ]:
def build_base_cnn():
    model = Sequential(
        [
            Input(shape=(128, 128, 3)),
            Rescaling(1.0 / 255),
            Conv2D(8, (3, 3), padding="same", activation="relu"),
            MaxPooling2D((2, 2)),
            Conv2D(16, (3, 3), padding="same", activation="relu"),
            MaxPooling2D((2, 2)),
            Conv2D(32, (3, 3), padding="same", activation="relu"),
            MaxPooling2D((2, 2)),
            Flatten(),
            Dense(len(cs.CLASSES), activation="softmax"),
        ],
        name="cnn_base_3scenes",
    )
    model.compile(
        loss="categorical_crossentropy",
        optimizer=Adam(learning_rate=1e-3),
        metrics=["accuracy"],
    )
    return model


def conv_block(x, filters):
    x = Conv2D(filters, (3, 3), padding="same", use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return MaxPooling2D((2, 2))(x)


def build_improved_cnn():
    inputs = Input(shape=(128, 128, 3))
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.08)(x)
    x = layers.RandomZoom(0.10)(x)
    x = layers.RandomContrast(0.10)(x)
    x = Rescaling(1.0 / 255)(x)
    x = conv_block(x, 32)
    x = conv_block(x, 64)
    x = conv_block(x, 128)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.4)(x)
    outputs = Dense(len(cs.CLASSES), activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="cnn_improved_3scenes")
    model.compile(
        loss="categorical_crossentropy",
        optimizer=Adam(learning_rate=1e-3),
        metrics=["accuracy"],
    )
    return model

## Entrenamiento o carga desde caché

In [ ]:
def fit_base(model):
    history, seconds = cs.train_keras_model(
        model, X_train, y_train, X_val, y_val, epochs=15, batch_size=32
    )
    return model, history, seconds


def evaluate_base(model, history, training_seconds):
    predicted_idx, probabilities, inference_ms = cs.predict_keras(model, X_test)
    predicted = np.array(cs.CLASSES)[predicted_idx]
    metrics = cs.compute_metrics(
        name="CNN base",
        family="CNN",
        y_true=y_test_labels,
        y_pred=predicted,
        y_prob=probabilities,
        params=model.count_params(),
        trainable_params=sum(np.prod(weight.shape) for weight in model.trainable_weights),
        training_seconds=training_seconds,
        inference_ms=inference_ms,
        epochs=len(history.history["loss"]),
        hyperparameters={
            "filters": [8, 16, 32],
            "optimizer": "Adam",
            "lr": 1e-3,
            "batch_size": 32,
            "epochs": 15,
            "augmentation": False,
        },
        notes="Sequential replica of the professor's base notebook (Adam decay removed in Keras 3).",
    )
    cs.save_results(
        "cnn/base",
        metrics,
        y_test_labels,
        predicted,
        probabilities,
        test_paths=test_paths,
        history=history,
        model=model,
    )
    return metrics


def fit_improved(model):
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5, verbose=1),
    ]
    history, seconds = cs.train_keras_model(
        model,
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=60,
        batch_size=32,
        callbacks=callbacks,
        class_weight=class_weight,
    )
    return model, history, seconds


def evaluate_improved(model, history, training_seconds):
    predicted_idx, probabilities, inference_ms = cs.predict_keras(model, X_test)
    predicted = np.array(cs.CLASSES)[predicted_idx]
    metrics = cs.compute_metrics(
        name="CNN mejorada",
        family="CNN",
        y_true=y_test_labels,
        y_pred=predicted,
        y_prob=probabilities,
        params=model.count_params(),
        trainable_params=sum(np.prod(weight.shape) for weight in model.trainable_weights),
        training_seconds=training_seconds,
        inference_ms=inference_ms,
        epochs=len(history.history["loss"]),
        hyperparameters={
            "filters": [32, 64, 128],
            "optimizer": "Adam",
            "lr": 1e-3,
            "batch_size": 32,
            "epochs_max": 60,
            "augmentation": True,
            "dropout": 0.4,
            "early_stopping": 12,
            "class_weight": True,
        },
        notes="CNN with BatchNorm, GAP, Dropout, augmentation and class weights.",
    )
    cs.save_results(
        "cnn/mejorada",
        metrics,
        y_test_labels,
        predicted,
        probabilities,
        test_paths=test_paths,
        history=history,
        model=model,
    )
    return metrics

In [ ]:
model_base, history_base, metrics_base, retrained_base = cs.load_or_train(
    "cnn/base",
    build_model=build_base_cnn,
    fit_model=fit_base,
    evaluate_model=evaluate_base,
    force=FORCE_RETRAIN,
)
print("Accuracy en test:", round(metrics_base["accuracy"], 4))

In [ ]:
model_improved, history_improved, metrics_improved, retrained_improved = cs.load_or_train(
    "cnn/mejorada",
    build_model=build_improved_cnn,
    fit_model=fit_improved,
    evaluate_model=evaluate_improved,
    force=FORCE_RETRAIN,
)
print("Accuracy en test:", round(metrics_improved["accuracy"], 4))

In [ ]:
model_base.summary()
print(f"\nParámetros de la CNN base: {model_base.count_params():,}")

model_improved.summary()
print(f"\nParámetros de la CNN mejorada: {model_improved.count_params():,}")

## Comparación de las dos variantes

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "model": metrics_base["name"],
            "accuracy": metrics_base["accuracy"],
            "f1_macro": metrics_base["f1_macro"],
            "parameters": metrics_base["parameters"],
            "training_seconds": metrics_base["training_seconds"],
        },
        {
            "model": metrics_improved["name"],
            "accuracy": metrics_improved["accuracy"],
            "f1_macro": metrics_improved["f1_macro"],
            "parameters": metrics_improved["parameters"],
            "training_seconds": metrics_improved["training_seconds"],
        },
    ]
)
display = comparison.rename(
    columns={
        "model": "Modelo",
        "accuracy": "Accuracy",
        "f1_macro": "F1 macro",
        "parameters": "Parámetros",
        "training_seconds": "Entrenamiento (s)",
    }
)
display.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
positions = np.arange(len(comparison))
width = 0.36
ax.bar(positions - width / 2, comparison["accuracy"], width, label="Accuracy")
ax.bar(positions + width / 2, comparison["f1_macro"], width, label="F1 macro")
ax.set_xticks(positions, comparison["model"])
ax.set_ylim(0, 1.05)
ax.set_title("CNN base vs CNN mejorada")
ax.legend()
for index, (accuracy, f1_value) in enumerate(
    zip(comparison["accuracy"], comparison["f1_macro"])
):
    ax.text(index - width / 2, accuracy + 0.01, f"{accuracy:.3f}", ha="center", fontsize=9)
    ax.text(index + width / 2, f1_value + 0.01, f"{f1_value:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## Observaciones

* La CNN base es deliberadamente simple y sin regularización, como en el
  cuaderno del profesor, por lo que se espera que sobreajuste (la accuracy de
  entrenamiento sube por encima de la de validación). Los números exactos
  quedan en la tabla de la sección anterior y en
  `resultados/cnn/base/metrics.json`.
* La CNN mejorada ataca ese sobreajuste con más filtros por bloque,
  `BatchNormalization`, `Dropout`, `GlobalAveragePooling`, augmentación y
  `class_weight`, y añade callbacks para parar a tiempo y reducir el learning
  rate. El coste es un modelo con más parámetros y un entrenamiento más largo
  en CPU.
* El flag `FORCE_RETRAIN = False` permite reutilizar los modelos ya guardados
  y evitar ese coste al revisar el cuaderno.
* La comparación completa con DenseNet121 y con los modelos de machine
  learning (IC 95 % por bootstrap, McNemar, matrices de confusión, curvas)
  está en `reto-1.ipynb` y en `resultados/comparacion/`.